In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
import scipy.stats as stats 



In [1]:
def matriz_corr(dados, variaveis, tamanho):

    """Rotina para plotar a matriz de correlação das variáveis de interesse. 
       A rotina toma 2 argumentos de entrada: 
        1 - 'dados': é o dataframe contendo as variáveis de interesse, onde nas linhas devem estar as observações e nas colunas as variáveis
        2 - 'variaveis': são as variáveis que se deseja analizar a correlação """
    
    correlacao = dados[variaveis].corr()
    
    figura, axes = plt.subplots(len(correlacao), len(correlacao), figsize = tamanho)

    for i in range(0, len(variaveis)):
        for j in range(0, len(variaveis)):

            if i > j: 
                sns.regplot(ax = axes[i,j], data = dados, x = dados[variaveis[j]],y = dados[variaveis[i]])
                if i != len(variaveis)-1:
                    axes[i,j].set(xticks = ([]), xlabel = '')
                if j != 0:
                    axes[i,j].set( yticks = ([]), ylabel = '')

            elif i ==j:
                sns.histplot(ax = axes [i,j], data = dados, x = variaveis[i], bins = 50, kde = True)
                if j != len(variaveis)-1:
                    axes[i,j].set(xticks = ([]), xlabel = '')
                else:
                    axes[i,j].set_xlabel(variaveis[i])
                if i != 0:
                    axes[i,j].set( yticks = ([]), ylabel = '')
                else:
                    axes[i,j].set_ylabel(variaveis[i])
                
            else:
                hist = axes[i,j].text(0.3, 0.5, correlacao.iloc[i,j].round(3))
                axes[i,j].set(xticks = ([]), xlabel = '')
                axes[i,j].set( yticks = ([]), ylabel = '')


In [1]:
def resumo(dados, class1, variaveis, tamanho):

    """Rotina para analizar os parâmetros decritivos de um grupo de variáveis de interesse. 
        A rotina toma 4 argumentos de entrada: 
         1 - 'dados': é o dataframe contendo as variáveis de interesse, onde nas linhas devem estar as observações e nas colunas as variáveis
         2 - 'class1': é a primeira classificação para o tratamento dos dados
         3 - 'variáveis: são as variáveis que serão analizadas
         4 - 'tamanho': tamanho da figura de saída 
         
         Se o dataframe for uma planilha de dados de um banco, por exemplo, aonde as linhas são os clientes (observações) e as colunas forem as seguntes;
         - sexo (F ou M)                                                       - QUALITATIVA
         - idade                                                               - QUANTITATIVA
         - empresa (público, privada ou autônomo)                              - QUALITATIVA
         - faixa salarial (nula, muito baixa, baixa, média, alta, muito alta)  - QUALITATIVA 
         - salário                                                             - QUANTITATIVA 
         - dívida cartão                                                       - QUANTITATIVA
         - poupança                                                            - QUANTITATIVA
         O parmetro 'class1' pode ser empresa. No caso os parâmetros descritivos serão segmentados por tipo de empresa. 
         O parâmetro 'variaveis' pode ser uma lista ['dívida cartão', 'poupança', 'idade']
         No caso em questão a rotina irá plotar 3 gráficos: 
         - 1º: a descrição de 'dívida cartão' segmentado por tipo de empresa 
         - 2º: a descrição de 'poupança' segmentado por tipo de empresa
         - 3º: a descrição de 'idade' segmentado por tipo de empresa
         Assim teremos 3 conjuntos de análise descritiva (um para cada variável quantitativa selecionada), segmentados de acordo com a variável 
         qualitativa selecionada"""

    figura,axes = plt.subplots(len(variaveis),2, figsize = tamanho, width_ratios=[1, 0.6])
    
    for i in range(0,len(variaveis)):
        
        descr = dados.groupby(class1)[variaveis[i]].describe()
        sns.boxplot(ax = axes[i,0], data = dados, x = class1, y = variaveis[i])
        
        
        axes[i,0].set_xlabel('')
        axes[i,1].table(cellText = descr.values.round(2), bbox = [0,0,1,1], colLabels = descr.columns, rowLabels = descr.index)
        
        axes[i,1].set(xticks = ([]), yticks = ([]))
        
        descr['var'] = descr['std']**2
        for j in descr.index:
            if descr.loc[j,'count']<=1:
                descr = descr.drop(j, axis = 0)
        

        var_pond = np.average(descr['var'], weights = descr['count'])
        var = dados[variaveis[i]].var()

        R2 = 1 - (var_pond/var)

        axes[i,0].legend([f'R2: {R2.round(3)}'])
    

In [1]:
def descritiva(input, nclasses):

  # plot da figura contendo 3 linhas e 2 colunas, com os respectivos tamanhos entre as figuras
  figura, axes = plt.subplots(3,2, figsize = (9,6), width_ratios = [1,0.7], height_ratios=[1,0.2,0.2])
  gs = axes[0,0].get_gridspec()
  [i.remove()  for i in axes[0:,1]]
  axes[0,1] = plt.subplot(gs[0:2,1])

  # plot do histograma com o número de classes = ao número de amostras
  principal = sns.histplot(ax = axes[0,0], data = input, bins = nclasses, kde = True,  color = 'lightpink')

  # cálculo da curva normal de distribuição
  x = np.linspace(min(input)-0.05*np.mean(input),max(input)+ 0.05*np.mean(input),500)
  y = stats.norm.pdf(x = x, loc = np.mean(input), scale = np.std(input))

  #plot da curva normal de distribuição de e curva de densidade de probabilidade
  principal.plot(x,y, color = 'k')

  #finalização do gráfico principal ( ajuste do eixo x e legenda dos gráficos)
  principal.set_xlim(left =min(input)-0.05*np.mean(input), right = max(input)+ 0.05*np.mean(input))
  principal.legend(['Probability Density','Normal Distribution','Results of Frequency'], fontsize = 6)
  principal.grid()
  principal.set_ylabel('')

  #plot do boxplot
  box_plot = sns.boxplot(ax = axes[1,0], data = input, color = 'crimson', orient = 'h')
  box_plot.set_xlim(left =min(input)-0.05*np.mean(input), right = max(input)+ 0.05*np.mean(input))
  box_plot.set(xticks = ([]), xlabel = (''))

  # cálculo e plot do intervalo de confiança
  z_valor = stats.norm.ppf(q = 0.975, loc = 0, scale = 1)
  LI = np.mean(input) - (z_valor * np.std(input)/(len(input)**0.5))
  LS = np.mean(input) + (z_valor * np.std(input)/(len(input)**0.5))
  axes[2,0].scatter([LI,LS], ['média','média'], marker = 'd', color = 'crimson')
  axes[2,0].plot([LI,LS], ['média','média'], color = 'crimson')
  axes[2,0].grid()
  axes[2,0].set_title('95% Confidence Interval')
  axes[2,0].set_yticks( ticks = [])

  # medidas de posição
  minimo = min(input)
  quantil1 = np.quantile(input,0.25)
  mediana = np.median(input)
  media = np.mean(input)
  quantil3 = np.quantile(input, 0.75)
  maximo = max(input)

  # medidas de dispersão
  assimetria = stats.skew(input)
  curtose = stats.kurtosis(input)
  variancia = np.var(input)
  desvio_pad = np.std(input)

  #intervalo de confiança para o desvio padrão
  chiquad_d = stats.chi2.ppf(q = 0.025, df = len(input) -1 )
  chiquad_e = stats.chi2.ppf(q = 0.975, df = len(input) -1 )
  LI_dp = (((len(input)-1)*np.var(input))/chiquad_e)**0.5
  LS_dp = (((len(input)-1)*np.var(input))/chiquad_d)**0.5

  # Caixa de informações com as medidas estatísticas relevantes e os intervalos de confiança
  axes[0,1].text(0.1,0.65,f'Position Measures\nMin: {minimo}\n1st Quartile: {round(quantil1,2)}\nMedian: {round(mediana,3)}\nMean: {round(media,3)}\n3st Qartile: {round(quantil3,2)}\nMax: {maximo}\nn: {len(input)}',\
                 backgroundcolor = 'lavenderblush')
  axes[0,1].text(0.1,0.35,f'Dispersion Measures\nAsymmetry: {round(assimetria,2)}\nKurtosis:\
   {round(curtose,2)}\nVariance: {round(variancia,2)}\nStandart Deviation: {round(desvio_pad,2)}', backgroundcolor = 'lavenderblush')
  axes[0,1].text(0.1,0.2 , f'Confidence Interval for the Mean\nMin: {round(LI,2)}        Max{round(LS,2)}', fontsize = 8, backgroundcolor = 'lavenderblush')
  axes[0,1].text(0.1,0.1, f'Confidence Interval for the StD\nMin: {round(LI_dp,2)}        Max{round(LS_dp,2)}', fontsize = 8, backgroundcolor = 'lavenderblush')
  axes[0,1].set_xticks(ticks = [])
  axes[0,1].set_yticks(ticks = [])

  plt.suptitle('Descriptive Measures', fontsize = 20)

In [ ]:
def CEP(input):

    resumo = input.describe()
    figura, axes = plt.subplot(1,1, figsize = (10,10))

    sns.pointplot(ax = axes, data = input)

    LIC = 
    LSC = 